**Celda 1 — Instalación**

In [ ]:
!pip -q install gliner pandas numpy tqdm transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.8/207.8 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 96.9 MB/s eta 0:00:00


**Celda 2 — Imports y configuración**

In [ ]:
import re
import gc
import json
import torch
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm
from google.colab import files
from gliner import GLiNER

pd.set_option("display.max_colwidth", 300)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

OUTPUT_DIR = Path("/content/outputs_03_gliner_embargos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARTIAL_DIR = OUTPUT_DIR / "reportes_embargos_gliner"
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de salida:", OUTPUT_DIR)

Dispositivo: cuda
Carpeta de salida: /content/outputs_03_gliner_embargos


**Antes de correr, liberar memoria**

In [ ]:
import gc
import torch

try:
    del model
except NameError:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memoria liberada.")

Memoria liberada.


**Celda 3 — Subir CSV de los 15 embargos**

In [ ]:
uploaded = files.upload()

csv_files = [name for name in uploaded.keys() if name.endswith(".csv")]

if not csv_files:
    raise FileNotFoundError("No se subió ningún archivo CSV.")

CSV_PATH = csv_files[0]
print("CSV cargado:", CSV_PATH)

Saving embargos_15_mejores_legibles.csv to embargos_15_mejores_legibles.csv
CSV cargado: embargos_15_mejores_legibles.csv


**Celda 4 — Cargar dataset y detectar columna de texto**

In [ ]:
df = pd.read_csv(CSV_PATH)

print("Filas:", len(df))
print("Columnas:")
print(df.columns.tolist())

display(df.head(3))

Filas: 15
Columnas:
['id', 'nombre', 'clasificacion', 'ocr_quality_score', 'score_seleccion_gliner', 'ocr_estado', 'len_despues', 'palabras_despues', 'porcentaje_reduccion', 'cantidad_indicadores_importantes', 'tiene_dni', 'tiene_cuil_cuit', 'tiene_cbu_cvu', 'tiene_expediente', 'tiene_monto', 'tiene_fecha', 'tiene_alias', 'tiene_banco', 'tiene_caratula', 'tiene_juzgado_secretaria', 'texto_limpieza_avanzada', 'texto_limpio_avanzado_preview_500']


,id,nombre,clasificacion,ocr_quality_score,score_seleccion_gliner,ocr_estado,len_despues,palabras_despues,porcentaje_reduccion,cantidad_indicadores_importantes,...,tiene_cbu_cvu,tiene_expediente,tiene_monto,tiene_fecha,tiene_alias,tiene_banco,tiene_caratula,tiene_juzgado_secretaria,texto_limpieza_avanzada,texto_limpio_avanzado_preview_500
0,536703,Embargo - usuario,Embargo,0.962489,0.979256,legible,4418,712,0.001130,8,...,False,True,True,True,False,True,True,True,"(Ue\n\n413/26, 09-18 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA- NOTIFICACIONES ELECTRÓNICAS\nUsuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar\nOrganismo: TRIBUNAL DEL TRABAJO N? 5 - QUILMES\ncarátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LA...","(Ue\n\n413/26, 09-18 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA- NOTIFICACIONES ELECTRÓNICAS\nUsuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar\nOrganismo: TRIBUNAL DEL TRABAJO N? 5 - QUILMES\ncarátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LA..."
1,542140,Embargo - usuario,Embargo,0.955085,0.975095,legible,4447,716,0.002020,8,...,False,True,True,True,False,True,True,True,"Poder Judicial de la Nación\njuzgado CIVIL 92\n\n29921/2018\n\nIncidente N* 1 - ACTOR: GONZALES\nGONZALES, JUANA JAQUELINE Y OTRO\nDEMANDADO: GUTIERREZ MONTES, ELOY\nES mercadO E ECUCION DE ALIMENTOS - INCIDENTE\nlibre\n\n1.3 MAR 2026 nn\npPCION CABA, 09 de Febrero de 2026.-\nRECE\n\nSEÑOR DIREC...","Poder Judicial de la Nación\njuzgado CIVIL 92\n\n29921/2018\n\nIncidente N* 1 - ACTOR: GONZALES\nGONZALES, JUANA JAQUELINE Y OTRO\nDEMANDADO: GUTIERREZ MONTES, ELOY\nES mercadO E ECUCION DE ALIMENTOS - INCIDENTE\nlibre\n\n1.3 MAR 2026 nn\npPCION CABA, 09 de Febrero de 2026.-\nRECE\n\nSEÑOR DIREC..."
2,546228,Embargo - usuario,Embargo,0.953850,0.974349,legible,4826,771,0.002687,8,...,True,True,True,True,False,True,True,True,"a 1+\n\nPoder Judicial de la Nación\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\n\ns/ALIMENTOS\npoa Buenos Aires, de marzo de 2026.- MEG\n\nTy Z OFICIO REITERATORIO\n\nMercado Pago\nAssetmanagement SA\n\nS / D\n\nTengo el agrado de dirigirme a Ud. en los autos...","a 1+\n\nPoder Judicial de la Nación\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\n\ns/ALIMENTOS\npoa Buenos Aires, de marzo de 2026.- MEG\n\nTy Z OFICIO REITERATORIO\n\nMercado Pago\nAssetmanagement SA\n\nS / D\n\nTengo el agrado de dirigirme a Ud. en los autos..."


In [ ]:
def detectar_columna_texto(df):
    candidatos_prioritarios = [
        "texto_limpieza_avanzada",
        "texto_limpio_avanzado",
        "texto_limpio",
        "texto_base",
        "texto",
        "texto_ocr",
        "contenido",
    ]

    for col in candidatos_prioritarios:
        if col in df.columns:
            return col

    # Si no encuentra por nombre, toma la columna object con mayor longitud mediana
    object_cols = df.select_dtypes(include=["object"]).columns.tolist()

    if not object_cols:
        raise ValueError("No se encontró ninguna columna de texto tipo object.")

    longitudes = {}

    for col in object_cols:
        longitudes[col] = df[col].fillna("").astype(str).str.len().median()

    return max(longitudes, key=longitudes.get)


TEXT_COL = detectar_columna_texto(df)

if "id" in df.columns:
    ID_COL = "id"
else:
    ID_COL = "_row_id"
    df[ID_COL] = range(len(df))

print("Columna de texto detectada:", TEXT_COL)
print("Columna ID:", ID_COL)

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

df["chars"] = df[TEXT_COL].str.len()
df["palabras"] = df[TEXT_COL].str.split().str.len()

display(df[[ID_COL, TEXT_COL, "chars", "palabras"]].head())

Columna de texto detectada: texto_limpieza_avanzada
Columna ID: id


,id,texto_limpieza_avanzada,chars,palabras
0,536703,"(Ue\n\n413/26, 09-18 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA- NOTIFICACIONES ELECTRÓNICAS\nUsuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar\nOrganismo: TRIBUNAL DEL TRABAJO N? 5 - QUILMES\ncarátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LA...",4418,674
1,542140,"Poder Judicial de la Nación\njuzgado CIVIL 92\n\n29921/2018\n\nIncidente N* 1 - ACTOR: GONZALES\nGONZALES, JUANA JAQUELINE Y OTRO\nDEMANDADO: GUTIERREZ MONTES, ELOY\nES mercadO E ECUCION DE ALIMENTOS - INCIDENTE\nlibre\n\n1.3 MAR 2026 nn\npPCION CABA, 09 de Febrero de 2026.-\nRECE\n\nSEÑOR DIREC...",4447,684
2,546228,"a 1+\n\nPoder Judicial de la Nación\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\n\ns/ALIMENTOS\npoa Buenos Aires, de marzo de 2026.- MEG\n\nTy Z OFICIO REITERATORIO\n\nMercado Pago\nAssetmanagement SA\n\nS / D\n\nTengo el agrado de dirigirme a Ud. en los autos...",4826,748
3,531044,"18/2/26, 10:49 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA - NOTIFICACIONES ELECTRÓNICAS dl 2)\n\nscba.gov.al\n\n7 Presentaciones y\n\nÑ Notificaciones Electrónicas\nA SUPREMA CORTE DE JUSTICIA\n\n<< aaa PROVINCIA DE BUENOS AIRES\n\nIsuario Conectado:Omar Roberto Garcia - Acceso...",5060,767
4,546230,"+ 19 MAR 2026 |\n\n1\n\nRECEPCION\n\nPoder Judicial de la Nación\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\ns/ALIMENTOS\n\nBuenos Aires, de marzo de 2026.- MEG\nOFICIO REIFERATORIO\n\nMercado Pago\nAssetmanagement SA\nS / D\n\nTengo el agrado de dirigirme a ...",4991,786


**Celda 5 — Labels para embargos**

Acá usamos labels específicos para embargos. Hay dos versiones:

*    `embargo_simple:` menos etiquetas, menos confusión.
*    `embargo_detallado:` más completa, pero puede generar más ruido.

In [ ]:
LABELS_EMBARGO_SIMPLE = [
    "monto",
    "destino de transferencia",
    "numero de expediente",
    "caratula",
    "persona",
    "DNI",
    "CUIL",
    "CUIT",
    "CBU",
    "banco",
]

LABELS_EMBARGO_DETALLADO = [
    "monto de embargo",
    "destino de transferencia",
    "numero de expediente",
    "caratula",
    "persona",
    "persona embargada",
    "persona solicitante",
    "DNI",
    "CUIL",
    "CUIT",
    "CBU",
    "CVU",
    "banco",
    "cuenta bancaria",
    "alias bancario",
    "juzgado",
    "secretaria",
    "fecha",
]

LABEL_SETS = {
    "embargo_simple": LABELS_EMBARGO_SIMPLE,
    "embargo_detallado": LABELS_EMBARGO_DETALLADO,
}

print("Labels simple:", LABELS_EMBARGO_SIMPLE)
print("Labels detallado:", LABELS_EMBARGO_DETALLADO)

Labels simple: ['monto', 'destino de transferencia', 'numero de expediente', 'caratula', 'persona', 'DNI', 'CUIL', 'CUIT', 'CBU', 'banco']
Labels detallado: ['monto de embargo', 'destino de transferencia', 'numero de expediente', 'caratula', 'persona', 'persona embargada', 'persona solicitante', 'DNI', 'CUIL', 'CUIT', 'CBU', 'CVU', 'banco', 'cuenta bancaria', 'alias bancario', 'juzgado', 'secretaria', 'fecha']


**Celda 6 — Configuración de modelos y prueba chica**

Primero probamos solo medium para confirmar que todo funcione.

In [ ]:
MODELOS_A_PROBAR = [
    "urchade/gliner_multi_pii-v1",
]

THRESHOLDS_A_PROBAR = [
    0.35,
]

LABELS_A_PROBAR = [
    "embargo_simple",
]

ESTRATEGIAS_A_PROBAR = [
    "chunks",
]

MAX_TOKENS_CHUNK = 300
OVERLAP_TOKENS = 50

print("Modelos:", MODELOS_A_PROBAR)
print("Thresholds:", THRESHOLDS_A_PROBAR)
print("Labels:", LABELS_A_PROBAR)
print("Estrategias:", ESTRATEGIAS_A_PROBAR)

Modelos: ['urchade/gliner_multi_pii-v1']
Thresholds: [0.35]
Labels: ['embargo_simple']
Estrategias: ['chunks']


**Celda 7 — Funciones auxiliares**

In [ ]:
def limpiar_nombre_modelo(model_name):
    return (
        model_name
        .replace("/", "__")
        .replace("-", "_")
        .replace(".", "_")
    )


def contar_tokens(texto, tokenizer):
    if not isinstance(texto, str):
        texto = str(texto)

    try:
        token_ids = tokenizer.encode(texto, add_special_tokens=False)
        return len(token_ids)
    except Exception:
        return np.nan


def crear_chunks_por_tokens(texto, tokenizer, max_tokens=300, overlap_tokens=50):
    """
    Divide el texto en chunks por tokens.
    Esto evita truncamiento en textos largos.
    """
    if not isinstance(texto, str):
        texto = str(texto)

    texto = texto.strip()

    if not texto:
        return []

    if overlap_tokens >= max_tokens:
        raise ValueError("overlap_tokens debe ser menor que max_tokens.")

    token_ids = tokenizer.encode(texto, add_special_tokens=False)

    chunks = []
    start = 0
    chunk_id = 0

    while start < len(token_ids):
        end = min(start + max_tokens, len(token_ids))
        token_slice = token_ids[start:end]

        try:
            chunk_text = tokenizer.decode(token_slice, skip_special_tokens=True)
        except TypeError:
            chunk_text = tokenizer.decode(token_slice)

        chunks.append({
            "chunk_id": chunk_id,
            "token_start": start,
            "token_end": end,
            "n_tokens_chunk": len(token_slice),
            "texto_chunk": chunk_text,
        })

        if end >= len(token_ids):
            break

        start = end - overlap_tokens
        chunk_id += 1

    return chunks


def contexto_entidad(texto, start, end, ventana=180):
    if not isinstance(texto, str):
        return ""

    if start is None or end is None:
        return texto[:500]

    try:
        start = int(start)
        end = int(end)
    except Exception:
        return texto[:500]

    inicio = max(0, start - ventana)
    fin = min(len(texto), end + ventana)

    return texto[inicio:fin]


def normalizar_entidad(ent):
    """
    GLiNER suele devolver:
    {
        'text': ...,
        'label': ...,
        'score': ...,
        'start': ...,
        'end': ...
    }

    Esta función protege por si alguna clave cambia.
    """
    return {
        "entidad_texto": ent.get("text", ent.get("span", None)),
        "entidad_label": ent.get("label", None),
        "score": ent.get("score", None),
        "start": ent.get("start", None),
        "end": ent.get("end", None),
    }


def extraer_directo(row, model, model_name, tokenizer, labels, labels_name, threshold):
    texto = str(row[TEXT_COL])
    row_id = row[ID_COL]

    registros = []

    try:
        n_tokens = contar_tokens(texto, tokenizer)

        entidades = model.predict_entities(
            texto,
            labels,
            threshold=threshold
        )

        if not entidades:
            registros.append({
                "id": row_id,
                "modelo": model_name,
                "labels_set": labels_name,
                "threshold": threshold,
                "estrategia": "directo",
                "chunk_id": None,
                "texto_chunk_chars": len(texto),
                "gliner_tokens": n_tokens,
                "entidad_texto": None,
                "entidad_label": "SIN_ENTIDADES",
                "score": None,
                "start": None,
                "end": None,
                "texto_contexto": texto[:500],
                "error": None,
            })
            return registros

        for ent in entidades:
            ent_norm = normalizar_entidad(ent)

            registros.append({
                "id": row_id,
                "modelo": model_name,
                "labels_set": labels_name,
                "threshold": threshold,
                "estrategia": "directo",
                "chunk_id": None,
                "texto_chunk_chars": len(texto),
                "gliner_tokens": n_tokens,
                **ent_norm,
                "texto_contexto": contexto_entidad(
                    texto,
                    ent_norm["start"],
                    ent_norm["end"]
                ),
                "error": None,
            })

    except Exception as e:
        registros.append({
            "id": row_id,
            "modelo": model_name,
            "labels_set": labels_name,
            "threshold": threshold,
            "estrategia": "directo",
            "chunk_id": None,
            "texto_chunk_chars": len(texto),
            "gliner_tokens": None,
            "entidad_texto": None,
            "entidad_label": "ERROR",
            "score": None,
            "start": None,
            "end": None,
            "texto_contexto": texto[:500],
            "error": str(e),
        })

    return registros


def extraer_chunks(row, model, model_name, tokenizer, labels, labels_name, threshold):
    texto = str(row[TEXT_COL])
    row_id = row[ID_COL]

    registros = []

    try:
        chunks = crear_chunks_por_tokens(
            texto,
            tokenizer,
            max_tokens=MAX_TOKENS_CHUNK,
            overlap_tokens=OVERLAP_TOKENS,
        )

        if not chunks:
            registros.append({
                "id": row_id,
                "modelo": model_name,
                "labels_set": labels_name,
                "threshold": threshold,
                "estrategia": "chunks",
                "chunk_id": None,
                "texto_chunk_chars": 0,
                "gliner_tokens": 0,
                "entidad_texto": None,
                "entidad_label": "SIN_TEXTO",
                "score": None,
                "start": None,
                "end": None,
                "texto_contexto": "",
                "error": None,
            })
            return registros

        encontro_entidades = False

        for chunk in chunks:
            chunk_text = chunk["texto_chunk"]

            entidades = model.predict_entities(
                chunk_text,
                labels,
                threshold=threshold
            )

            if entidades:
                encontro_entidades = True

            for ent in entidades:
                ent_norm = normalizar_entidad(ent)

                registros.append({
                    "id": row_id,
                    "modelo": model_name,
                    "labels_set": labels_name,
                    "threshold": threshold,
                    "estrategia": "chunks",
                    "chunk_id": chunk["chunk_id"],
                    "texto_chunk_chars": len(chunk_text),
                    "gliner_tokens": chunk["n_tokens_chunk"],
                    **ent_norm,
                    "texto_contexto": contexto_entidad(
                        chunk_text,
                        ent_norm["start"],
                        ent_norm["end"]
                    ),
                    "error": None,
                })

        if not encontro_entidades:
            registros.append({
                "id": row_id,
                "modelo": model_name,
                "labels_set": labels_name,
                "threshold": threshold,
                "estrategia": "chunks",
                "chunk_id": None,
                "texto_chunk_chars": len(texto),
                "gliner_tokens": contar_tokens(texto, tokenizer),
                "entidad_texto": None,
                "entidad_label": "SIN_ENTIDADES",
                "score": None,
                "start": None,
                "end": None,
                "texto_contexto": texto[:500],
                "error": None,
            })

    except Exception as e:
        registros.append({
            "id": row_id,
            "modelo": model_name,
            "labels_set": labels_name,
            "threshold": threshold,
            "estrategia": "chunks",
            "chunk_id": None,
            "texto_chunk_chars": len(texto),
            "gliner_tokens": None,
            "entidad_texto": None,
            "entidad_label": "ERROR",
            "score": None,
            "start": None,
            "end": None,
            "texto_contexto": texto[:500],
            "error": str(e),
        })

    return registros

**Celda 8 — Cargar un modelo y calcular tokens**

Esta celda carga el primer modelo configurado.

In [ ]:
model_name = MODELOS_A_PROBAR[0]

print("Cargando modelo:", model_name)

model = GLiNER.from_pretrained(model_name)
model = model.to(DEVICE)

tokenizer = model.data_processor.transformer_tokenizer

print("Modelo cargado:", model_name)

Cargando modelo: urchade/gliner_multi_pii-v1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

Modelo cargado: urchade/gliner_multi_pii-v1


In [ ]:
df_tokens = df.copy()

df_tokens["gliner_tokens"] = df_tokens[TEXT_COL].apply(
    lambda x: contar_tokens(str(x), tokenizer)
)

df_tokens["riesgo_truncamiento"] = df_tokens["gliner_tokens"] > 300

display(
    df_tokens[[ID_COL, "chars", "palabras", "gliner_tokens", "riesgo_truncamiento"]]
    .sort_values("gliner_tokens", ascending=False)
)

,id,chars,palabras,gliner_tokens,riesgo_truncamiento
13,534570,6640,1001,2026,True
8,529790,6379,1018,1903,True
14,538081,5697,904,1736,True
6,536741,5445,866,1588,True
3,531044,5060,767,1566,True
9,546250,4971,789,1545,True
4,546230,4991,786,1542,True
7,547358,5017,752,1480,True
2,546228,4826,748,1479,True
5,546231,4754,752,1414,True


**Celda 9 — Probar extracción en 1 embargo primero**

Esto es para revisar rápido si el modelo está detectando algo razonable.

In [ ]:
idx_prueba = 0

row = df.iloc[idx_prueba]

labels_name = "embargo_simple"
labels = LABEL_SETS[labels_name]
threshold = 0.35

texto_prueba = str(row[TEXT_COL])

print("ID:", row[ID_COL])
print("Caracteres:", len(texto_prueba))
print("Tokens:", contar_tokens(texto_prueba, tokenizer))
print("Labels:", labels)
print("\nTexto preview:\n")
print(texto_prueba[:1200])

ID: 536703
Caracteres: 4418
Tokens: 1355
Labels: ['monto', 'destino de transferencia', 'numero de expediente', 'caratula', 'persona', 'DNI', 'CUIL', 'CUIT', 'CBU', 'banco']

Texto preview:

(Ue

413/26, 09-18 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA- NOTIFICACIONES ELECTRÓNICAS
Usuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar
Organismo: TRIBUNAL DEL TRABAJO N? 5 - QUILMES
carátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO
Número de causa: 18645
Tipo de notificación: OFICIO REQUIRIENDO
Destinatarios: 23137661349 notificaciones.scba.govar
Fecha notificación: 27/2/2026 08:27:57
Alta o disponibilidad 27/2/202608:28:06 10 AR 2008
Firma digital: Firma válida
Firmado y Notificado por: ZACARIAS Andrea Marcela. JUEZ - Certifigado

08:28:05
Elrmado por: ZACARIAS Andrea Marcela. JUEZ - Certificado Correcto. Fecha de Firma: 27/02/2026

08:28:04 DOS SANTOS Carlos Roberto. - Certificado Correcto. Fecha de Firma:
25/02/202

In [ ]:
entidades_prueba = model.predict_entities(
    texto_prueba[:4000],
    labels,
    threshold=threshold
)

print("Entidades detectadas:", len(entidades_prueba))

for ent in entidades_prueba:
    print(ent)

Entidades detectadas: 11
{'start': 128, 'end': 153, 'text': 'DOS SANTOS CARLOS ROBERTO', 'label': 'persona', 'score': 0.9408165216445923}
{'start': 252, 'end': 314, 'text': 'DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO', 'label': 'caratula', 'score': 0.8911951780319214}
{'start': 332, 'end': 337, 'text': '18645', 'label': 'numero de expediente', 'score': 0.40325042605400085}
{'start': 394, 'end': 405, 'text': '23137661349', 'label': 'destino de transferencia', 'score': 0.41950637102127075}
{'start': 576, 'end': 599, 'text': 'ZACARIAS Andrea Marcela', 'label': 'persona', 'score': 0.7207509279251099}
{'start': 643, 'end': 666, 'text': 'ZACARIAS Andrea Marcela', 'label': 'persona', 'score': 0.8622347116470337}
{'start': 734, 'end': 759, 'text': 'DOS SANTOS Carlos Roberto', 'label': 'persona', 'score': 0.9685788750648499}
{'start': 926, 'end': 988, 'text': 'DEFINA STELLA MARIS C/\nARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO', 'label': 'caratula', 'score': 0.4806753695011139}


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 809 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


**FORMA MAS LEGIBLE DEL RESULTADO**

In [ ]:
df_prueba_entidades = pd.DataFrame(entidades_prueba)

if len(df_prueba_entidades) == 0:
    print("No se detectaron entidades.")
else:
    columnas = ["label", "text", "score", "start", "end"]
    df_prueba_entidades = df_prueba_entidades[columnas].copy()
    df_prueba_entidades["score"] = df_prueba_entidades["score"].round(3)

    display(
        df_prueba_entidades
        .sort_values(["label", "score"], ascending=[True, False])
        .reset_index(drop=True)
    )

,label,text,score,start,end
0,caratula,DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO,0.891,252,314
1,caratula,DEFINA STELLA MARIS C/\nARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO,0.481,926,988
2,destino de transferencia,23137661349,0.420,394,405
3,numero de expediente,18645,0.403,332,337
4,numero de expediente,18645,0.356,1002,1007
5,persona,DOS SANTOS Carlos Roberto,0.969,734,759
6,persona,DOS SANTOS CARLOS ROBERTO,0.941,128,153
7,persona,Dr. Mario Daniel\nStolarczyk,0.913,1104,1131
8,persona,Dra. María Fernanda Sartori,0.913,1158,1185
9,persona,ZACARIAS Andrea Marcela,0.862,643,666


**Celda 10 — Ejecutar extracción configurada**

Esta es la celda importante. Con la configuración inicial corre:

*   15 embargos  
*   1 modelo  
*   1 threshold  
*   labels simples  
*   estrategia chunks  

In [ ]:
todos_los_resultados = []

for model_name in MODELOS_A_PROBAR:
    print("\n" + "=" * 80)
    print("Modelo:", model_name)
    print("=" * 80)

    model = GLiNER.from_pretrained(model_name)
    model = model.to(DEVICE)
    tokenizer = model.data_processor.transformer_tokenizer

    resultados_modelo = []

    for labels_name in LABELS_A_PROBAR:
        labels = LABEL_SETS[labels_name]

        for threshold in THRESHOLDS_A_PROBAR:
            for estrategia in ESTRATEGIAS_A_PROBAR:

                print(f"\nEjecutando: labels={labels_name} | threshold={threshold} | estrategia={estrategia}")

                for _, row in tqdm(df.iterrows(), total=len(df)):

                    if estrategia == "directo":
                        registros = extraer_directo(
                            row=row,
                            model=model,
                            model_name=model_name,
                            tokenizer=tokenizer,
                            labels=labels,
                            labels_name=labels_name,
                            threshold=threshold,
                        )

                    elif estrategia == "chunks":
                        registros = extraer_chunks(
                            row=row,
                            model=model,
                            model_name=model_name,
                            tokenizer=tokenizer,
                            labels=labels,
                            labels_name=labels_name,
                            threshold=threshold,
                        )

                    else:
                        raise ValueError(f"Estrategia no reconocida: {estrategia}")

                    resultados_modelo.extend(registros)
                    todos_los_resultados.extend(registros)

    df_modelo = pd.DataFrame(resultados_modelo)

    # Quitar duplicados exactos en chunks por overlap
    df_modelo = df_modelo.drop_duplicates(
        subset=[
            "id",
            "modelo",
            "labels_set",
            "threshold",
            "estrategia",
            "entidad_label",
            "entidad_texto",
        ],
        keep="first",
    )

    partial_path = PARTIAL_DIR / f"entidades_{limpiar_nombre_modelo(model_name)}_parcial.csv"
    df_modelo.to_csv(partial_path, index=False, encoding="utf-8-sig")

    print("Guardado parcial:", partial_path)

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Modelo: urchade/gliner_multi_pii-v1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]


Ejecutando: labels=embargo_simple | threshold=0.35 | estrategia=chunks


  0%|          | 0/15 [00:00<?, ?it/s]

Guardado parcial: /content/outputs_03_gliner_embargos/reportes_embargos_gliner/entidades_urchade__gliner_multi_pii_v1_parcial.csv


**Celda 11 — Unir resultados y guardar CSV principal**

In [ ]:
df_entidades = pd.DataFrame(todos_los_resultados)

if len(df_entidades) == 0:
    raise ValueError("No se generaron resultados.")

df_entidades = df_entidades.drop_duplicates(
    subset=[
        "id",
        "modelo",
        "labels_set",
        "threshold",
        "estrategia",
        "entidad_label",
        "entidad_texto",
    ],
    keep="first",
)

ENTIDADES_PATH = OUTPUT_DIR / "entidades_gliner_embargos_15.csv"

df_entidades.to_csv(
    ENTIDADES_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado:", ENTIDADES_PATH)
print("Filas:", len(df_entidades))

display(df_entidades.head(20))

Archivo guardado: /content/outputs_03_gliner_embargos/entidades_gliner_embargos_15.csv
Filas: 440


,id,modelo,labels_set,threshold,estrategia,chunk_id,texto_chunk_chars,gliner_tokens,entidad_texto,entidad_label,score,start,end,texto_contexto,error
0,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,939,300,DOS SANTOS CARLOS ROBERTO,persona,0.966278,127,152,"(Ue 413/26, 09-18 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA- NOTIFICACIONES ELECTRÓNICAS Usuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar Organismo: TRIBUNAL DEL TRABAJO N? 5 - QUILMES carátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO E...",None
1,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,939,300,DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO,caratula,0.921589,251,313,USTICIA- NOTIFICACIONES ELECTRÓNICAS Usuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar Organismo: TRIBUNAL DEL TRABAJO N? 5 - QUILMES carátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO Número de causa: 18645 Tipo de notificación: OFICIO R...,None
2,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,939,300,18645,numero de expediente,0.716898,331,336,O - 23137661349Q notificaciones.scba.govar Organismo: TRIBUNAL DEL TRABAJO N? 5 - QUILMES carátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO Número de causa: 18645 Tipo de notificación: OFICIO REQUIRIENDO Destinatarios: 23137661349 notificaciones.scba.govar Fecha notificaci...,None
3,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,939,300,23137661349 notificaciones.scba.govar,destino de transferencia,0.507768,393,430,DEL TRABAJO N? 5 - QUILMES carátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO Número de causa: 18645 Tipo de notificación: OFICIO REQUIRIENDO Destinatarios: 23137661349 notificaciones.scba.govar Fecha notificación: 27/2/2026 08:27:57 Alta o disponibilidad 27/2/202608:28:06...,None
4,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,939,300,ZACARIAS Andrea Marcela,persona,0.972130,575,598,137661349 notificaciones.scba.govar Fecha notificación: 27/2/2026 08:27:57 Alta o disponibilidad 27/2/202608:28:06 10 AR 2008 Firma digital: Firma válida Firmado y Notificado por: ZACARIAS Andrea Marcela. JUEZ - Certifigado 08:28:05 Elrmado por: ZACARIAS Andrea Marcela. JUEZ - Certificado Correc...,None
6,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,939,300,DOS SANTOS Carlos Roberto,persona,0.990230,731,756,rmado y Notificado por: ZACARIAS Andrea Marcela. JUEZ - Certifigado 08:28:05 Elrmado por: ZACARIAS Andrea Marcela. JUEZ - Certificado Correcto. Fecha de Firma: 27/02/2026 08:28:04 DOS SANTOS Carlos Roberto. - Certificado Correcto. Fecha de Firma: 25/02/2026 13:55:01 OFICIO DE embargo AMERCADO PA...,None
7,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,939,300,DEFINA STELLA MARIS,caratula,0.558497,920,939,"S Carlos Roberto. - Certificado Correcto. Fecha de Firma: 25/02/2026 13:55:01 OFICIO DE embargo AMERCADO PAGO SID.- Tengo el agrado de dirigirme a Usted en los autos caratulados "":DEFINA STELLA MARIS",None
9,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,1,967,300,N* Causa: 18645,numero de expediente,0.883877,198,213,"26 13:55:01 OFICIO DE embargo AMERCADO PAGO SID.- Tengo el agrado de dirigirme a Usted en los autos caratulados "":DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO"" - N* Causa: 18645, que tramita ante el Tribunal de Trabajo N* 5 del Departamento Judicial de Quilmes, a cargo del Dr. ...",None
10,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,1,967,300,Dr. Mario Daniel Stolarczyk,persona,0.977235,310,337,""":DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO"" - N* Causa: 18645, que tramita ante el Tribunal de Trabajo N* 5 del Departamento Judicial de Quilmes, a cargo del Dr. Mario Daniel Stolarczyk, secretaría a cargo de la Dra. María Fernanda Sartori, sito en calle Alvear nro. 838 per...",None
11,536703,

**Celda 12 — Marcar entidades sospechosas**

In [ ]:
def formato_valido_por_label(label, texto):
    if pd.isna(texto):
        return False

    label = str(label).lower().strip()
    texto = str(texto).strip()
    solo_digitos = re.sub(r"\D", "", texto)

    # Labels textuales: no tienen formato numérico fijo
    if label in ["persona", "caratula", "banco"]:
        return True

    if "dni" in label:
        return bool(re.fullmatch(r"\d{7,8}", solo_digitos))

    if "cuil" in label or "cuit" in label:
        return bool(re.fullmatch(r"(20|23|24|27|30|33|34)\d{9}", solo_digitos))

    if "cbu" in label or "cvu" in label:
        return bool(re.fullmatch(r"\d{22}", solo_digitos))

    if "monto" in label:
        tiene_numero = bool(re.search(r"\d", texto))
        tiene_indicio_monto = bool(re.search(r"\$|pesos|importe|monto", texto, flags=re.IGNORECASE))
        return tiene_numero and tiene_indicio_monto

    if "expediente" in label:
        return bool(re.search(r"\d", texto)) and len(texto) >= 3

    if "destino de transferencia" in label:
        # Debe parecer una cuenta, CBU/CVU, banco, alias o instrucción real de transferencia
        tiene_cbu_cvu = bool(re.search(r"\b(?:CBU|CVU)\b|\d(?:\D?\d){21}", texto, flags=re.IGNORECASE))
        tiene_banco = bool(re.search(r"banco|mercado\s*pago|cuenta|alias|transfer", texto, flags=re.IGNORECASE))
        parece_mail_o_notificacion = bool(re.search(r"notificaciones|scba|@|gov", texto, flags=re.IGNORECASE))

        if parece_mail_o_notificacion and not tiene_cbu_cvu:
            return False

        return tiene_cbu_cvu or tiene_banco

    return True


def requiere_revision_manual(row):
    label = str(row["entidad_label"]).lower()
    texto = str(row["entidad_texto"])
    score = row["score"]

    if pd.isna(score):
        return True

    # Score medio/bajo para revisión
    if score < 0.60:
        return True

    # Formato inválido según label
    if row["formato_valido_label"] == False:
        return True

    # Entidades con posible ruido OCR
    if row["posible_ruido_ocr"] == True:
        return True

    # Persona con cargo pegado
    if label == "persona" and re.search(r"\bjuez\b|\bdra?\.?\b|\bsecretario\b", texto, flags=re.IGNORECASE):
        return True

    # Carátula demasiado corta
    if label == "caratula" and len(texto.split()) < 4:
        return True

    return False


df_entidades["formato_valido_label"] = df_entidades.apply(
    lambda row: formato_valido_por_label(row["entidad_label"], row["entidad_texto"]),
    axis=1
)

df_entidades["requiere_revision_manual"] = df_entidades.apply(
    requiere_revision_manual,
    axis=1
)

df_revision_rapida = df_entidades[
    ~df_entidades["entidad_label"].isin(["SIN_ENTIDADES", "SIN_TEXTO", "ERROR"])
].copy()

df_revision_rapida["score"] = df_revision_rapida["score"].round(3)

display(
    df_revision_rapida[
        [
            "id",
            "chunk_id",
            "entidad_label",
            "entidad_texto",
            "score",
            "formato_valido_label",
            "requiere_revision_manual",
            "texto_contexto",
        ]
    ]
    .sort_values(["requiere_revision_manual", "entidad_label", "score"], ascending=[False, True, True])
    .reset_index(drop=True)
)

,id,chunk_id,entidad_label,entidad_texto,score,formato_valido_label,requiere_revision_manual,texto_contexto
0,531044,4,CBU,30-70721665-0,0.501,False,True,"30-70721665-0) a la orden del juzgado y como perteneciente a estos autos, dentro del plazo de cinco días hábiles posteriores a la liquidación de los haberes correspondientes, por lo que, se ord"
1,547358,5,CBU,141038119503,0.562,False,True,"CVU: 0000003100087376033711 Número de operación de Mercado Pago 134597534609 Scanned with (BS CamScanner Detalle de operación Creada el 12 de enero - 10:16 hs Número de operación 141038119503 (bh () transferencia de dinero $ 150.000,00 por ""Varios"" [y] Aprobada (9) transferencia de Sebastian Le..."
2,531044,3,CBU,01401505275049509868834,0.851,False,True,"ROBERTO C/ ROLON GONZALEZ JULIO CESAR S/ COBRO DE HONORARIOS - Número: 6527 informando cuenta judicial N"" 027-509688/3 y CBU N* 01401505275049509868834. CUIT Poder Judicial 30- 70721665-0. El auto ordenatorio dice ""Avellaneda,17 de noviembre de 2025. Conforme lo pedido y hasta cubrir la suma de ..."
3,546231,3,CBU,2204 36324485570295202512231 00909208 0290075900204069109960,0.852,False,True,"e que cumpla con el embargo ordenado el día 30/9/25, hágase saber a la entidad oficiada que deberá depositar las sumas retenidas en la cuenta judicial: T"" 691 F* 996 DV 0, con CBU 2204 36324485570295202512231 00909208 0290075900204069109960. A ial in líbrese oficio digital. Hágase saber a la par..."
4,531044,4,CBU,CBU,0.941,False,True,"n de los haberes correspondientes, por lo que, se ordena proceder a la apertura de la cuenta judicial en cuestión, fecho la parte, deberá consignar los datos de la misma, número y CBU, en el oficio al empleador. A tales fines líbrese por secretaría en forma electrónica la pieza pertinente. Fecho..."
...,...,...,...,...,...,...,...,...
435,534570,4,persona,JOSE CESAR ROJAS,0.997,True,False,"reses, costas y costos. Librese oficio, respecto de JUAN MARIA ROJAS, (DNI N* 33.876.548) en MERCADO PAGO; MARIA DEL VALLE PEREZ (DNI N* 12.927.185) en el banco SUPERVIELLE S.A. y JOSE CESAR ROJAS (DNI N* 7.760.676) en el banco COMAF] SOCIEDAD ANONIMA a fin de trabar el embargo ordenado sobre la..."
436,531044,0,persona,Omar Roberto Garcia,0.998,True,False,CIA - NOTIFICACIONES ELECTRÓNICAS dl 2) scba.gov.al 7 Presentaciones y Ñ Notificaciones Electrónicas A SUPREMA CORTE DE JUSTICIA << aaa PROVINCIA DE BUENOS AIRES Isuario Conectado:Omar Roberto Garcia - Acceso anterior: 18/02/2026 09:39- Accesos anterior sólo lectura: 15/10/2025 08:23 errar Sesió...
437,534570,2,persona,Juan Maria ROJAS,0.998,True,False,"a trabar embargo sobre las cuentas bancarias, cuentas corrientes, cajas de ahorro, depósitos a plazo fijo, bonos o acciones de la co- demandado Juan Maria ROJAS, DNI N* 33.876.548, hasta cubrir la suma de $7,752.750,03 (pesos SIETE MILLONES SETECIENTOS CINCUENTA Y DOS MIL SETECIENTOS CINCUENTA C..."
438,534570,4,persona,JUAN MARIA ROJAS,0.998,True,False,"2.750,03 que racae en cada uno de los requeridos con más la suma de pesos $ 11.000.000 que se presupuestan para responder a intereses, costas y costos. Librese oficio, respecto de JUAN MARIA ROJAS, (DNI N* 33.876.548) en MERCADO PAGO; MARIA DEL VALLE PEREZ (DNI N* 12.927.185) en el banco SUPERVI..."


**Celda 13 — Comparativa de modelos / thresholds / estrategias**

In [ ]:
df_validas = df_entidades[
    ~df_entidades["entidad_label"].isin(["SIN_ENTIDADES", "SIN_TEXTO", "ERROR"])
].copy()

df_comparativa_mejorada = (
    df_validas
    .groupby(["modelo", "labels_set", "threshold", "estrategia"], dropna=False)
    .agg(
        total_entidades=("entidad_texto", "count"),
        documentos_cubiertos=("id", "nunique"),
        score_promedio=("score", "mean"),
        score_minimo=("score", "min"),
        score_maximo=("score", "max"),
        entidades_formato_valido=("formato_valido_label", "sum"),
        entidades_formato_invalido=("formato_valido_label", lambda x: (~x).sum()),
        entidades_requieren_revision=("requiere_revision_manual", "sum"),
        entidades_confiables=("requiere_revision_manual", lambda x: (~x).sum()),
    )
    .reset_index()
)

for col in ["score_promedio", "score_minimo", "score_maximo"]:
    df_comparativa_mejorada[col] = df_comparativa_mejorada[col].round(3)

display(df_comparativa_mejorada)

COMPARATIVA_MEJORADA_PATH = OUTPUT_DIR / "comparativa_mejorada_multi_pii_v1.csv"

df_comparativa_mejorada.to_csv(
    COMPARATIVA_MEJORADA_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Comparativa mejorada guardada en:", COMPARATIVA_MEJORADA_PATH)

,modelo,labels_set,threshold,estrategia,total_entidades,documentos_cubiertos,score_promedio,score_minimo,score_maximo,entidades_formato_valido,entidades_formato_invalido,entidades_requieren_revision,entidades_confiables
0,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,440,15,0.776,0.351,0.999,396,44,163,277


Comparativa mejorada guardada en: /content/outputs_03_gliner_embargos/comparativa_mejorada_multi_pii_v1.csv


In [ ]:
resumen_revision = (
    df_validas
    .groupby(["entidad_label", "formato_valido_label", "requiere_revision_manual"])
    .size()
    .reset_index(name="cantidad")
    .sort_values(["entidad_label", "requiere_revision_manual", "cantidad"], ascending=[True, False, False])
)

display(resumen_revision)

,entidad_label,formato_valido_label,requiere_revision_manual,cantidad
0,CBU,False,True,7
1,CBU,True,False,13
2,CUIL,False,True,2
3,CUIL,True,False,2
4,CUIT,False,True,2
5,CUIT,True,False,18
8,DNI,True,True,2
6,DNI,False,True,1
7,DNI,True,False,20
10,banco,True,True,18


el modelo funciona bien para personas, montos, CUIT, DNI, bancos y expediente. Donde más hay que revisar es destino de transferencia, CBU, monto textual y carátulas parciales.

**Para separar entidades confiables y entidades a revisar**  
**Entidades confiables**

In [ ]:
df_confiables = df_validas[
    df_validas["requiere_revision_manual"] == False
].copy()

display(
    df_confiables[
        ["id", "chunk_id", "entidad_label", "entidad_texto", "score", "texto_contexto"]
    ]
    .sort_values(["id", "entidad_label", "score"], ascending=[True, True, False])
    .reset_index(drop=True)
)

,id,chunk_id,entidad_label,entidad_texto,score,texto_contexto
0,524010,1,CBU,02900759-00225079500846,0.855543,"$25.0 presupuestada provisoriamente para responder intere costas.- Las sumas retenidas, deberán depositadas en la cuenta judicial LO 795, Fo 84, Dv 8 $4483175857H1202512040: CBU: 02900759-00225079500846, especialmente abierta al efecto en las actuaciones citadas y a la orden del Juez intervinie..."
1,524010,3,CUIT,33-71862856-9,0.952500,"nvertidos en cuota partes del Fondo común de inversión Poder Judicial de la Nación juzgado COMERCIAL 12 Mercado Fondos administrado por Mercado Pagol lA Eset Management S.A., CUIT 33-71862856-9, con domicilio] é la, Caseros 3039, Piso 2, Ciudad Autónoma de Buenos h rs y que fueran suceptibles de..."
2,524010,1,CUIT,NO 30-99903208-3,0.779093,"00225079500846, especialmente abierta al efecto en las actuaciones citadas y a la orden del Juez interviniente, en el banco de la Ciudad de Buenos Aires, Sucursal Tribunales, CUIT NO 30-99903208-3.- Se hace saber que la medida no se hará efectiva en caso que el titular de la cuenta sea persona e..."
3,524010,1,banco,Mercado Pago Asset Management S.,0.744620,". Dicha medida fue ordenada sob fondos presentes y futuros que tenga invertidos en dul partes del Fondo común de inversión Mercado | Fi administrado por Mercado Pago Asset Management S. demandado en autos hasta cubrir las sumas de $3.0b| en concepto de capital, con más la suma de $25.0 presupues..."
4,524010,2,banco,banco,0.739216,"so, caja de ahorro fueran salariales en los términos que impone el art. 124 de la ley 20.744, DL. 647/97 y las resoluciones del M.T. 644 y 790 360/2001, y el empleador no fuera el banco, no se deberá trabar la medida, debiendo informar la entidad el nombre o razón social del empleador y su domic..."
...,...,...,...,...,...,...
272,547393,4,persona,Bruno Mesa,0.892396,rrespondiente número de expediente y año en la solapa de consulta de expedientes.- Quedan autorizados a diligenciar este oficio: Nancy Rivero Acuña y/o Maria Julieta Teran y/o y/o Bruno Mesa y/o Guillon Candela y/o Nicolás de la Fuente y/o Micaela Prieto y/o quienes ellos designen.- Poder Judici...
273,547393,4,persona,Guillon Candela,0.882463,úmero de expediente y año en la solapa de consulta de expedientes.- Quedan autorizados a diligenciar este oficio: Nancy Rivero Acuña y/o Maria Julieta Teran y/o y/o Bruno Mesa y/o Guillon Candela y/o Nicolás de la Fuente y/o Micaela Prieto y/o quienes ellos designen.- Poder Judicial de la Nación...
274,547393,4,persona,Nicolás de la Fuente,0.719328,y año en la solapa de consulta de expedientes.- Quedan autorizados a diligenciar este oficio: Nancy Rivero Acuña y/o Maria Julieta Teran y/o y/o Bruno Mesa y/o Guillon Candela y/o Nicolás de la Fuente y/o Micaela Prieto y/o quienes ellos designen.- Poder Judicial de la Nación juzgado CIVIL 58 Si...
275,547393,2,persona,Fdo. María Di Filippo,0.712655,"Buenos Aires 19/12/2025: ""Agréguese. A idénticos fines y efectos que el oficio, cuyo diligenciamiento luce con el escrito en despacho, líbrese el oficio reiteratorio solicitado.""-Fdo. María Di Filippo, Jueza.- Se transcriben a sus efectos los siguientes artículos del Código Procesal en su parte..."


**Entidades confiables:** Son las que tienen buen score y formato razonable.

**Entidades a revisar**

In [ ]:
df_a_revisar = df_validas[
    df_validas["requiere_revision_manual"] == True
].copy()

display(
    df_a_revisar[
        ["id", "chunk_id", "entidad_label", "entidad_texto", "score", "formato_valido_label", "texto_contexto"]
    ]
    .sort_values(["entidad_label", "score"], ascending=[True, False])
    .reset_index(drop=True)
)

,id,chunk_id,entidad_label,entidad_texto,score,formato_valido_label,texto_contexto
0,529790,0,CBU,01100402-500990,0.948257,False,"IT. 20235539897, conforme informara en autos con fecha 04/11/25, a la cuenta de autos abierta en el banco Nación, Sucursal San Martin, que se identifica con el N? 9902829493, CBU. 01100402-500990"
1,531044,2,CBU,014015052750,0.943527,False,"EMA CORTE DE JUSTICIA - NOTIFICACIONES ELECTRÓNICAS ROBERTO C/ ROLON GONZALEZ JULIO CESAR S/ COBRO DE HONORARIOS - Número: 6527 informando cuenta judicial N"" 027-509688/3 y CBU N* 014015052750"
2,531044,4,CBU,CBU,0.941261,False,"n de los haberes correspondientes, por lo que, se ordena proceder a la apertura de la cuenta judicial en cuestión, fecho la parte, deberá consignar los datos de la misma, número y CBU, en el oficio al empleador. A tales fines líbrese por secretaría en forma electrónica la pieza pertinente. Fecho..."
3,546231,3,CBU,2204 36324485570295202512231 00909208 0290075900204069109960,0.851950,False,"e que cumpla con el embargo ordenado el día 30/9/25, hágase saber a la entidad oficiada que deberá depositar las sumas retenidas en la cuenta judicial: T"" 691 F* 996 DV 0, con CBU 2204 36324485570295202512231 00909208 0290075900204069109960. A ial in líbrese oficio digital. Hágase saber a la par..."
4,531044,3,CBU,01401505275049509868834,0.850708,False,"ROBERTO C/ ROLON GONZALEZ JULIO CESAR S/ COBRO DE HONORARIOS - Número: 6527 informando cuenta judicial N"" 027-509688/3 y CBU N* 01401505275049509868834. CUIT Poder Judicial 30- 70721665-0. El auto ordenatorio dice ""Avellaneda,17 de noviembre de 2025. Conforme lo pedido y hasta cubrir la suma de ..."
...,...,...,...,...,...,...,...
158,547358,1,persona,IMARESCO GRACIELA SILVIA C,0.385491,True,"IOS DE PROCESAMIENTO S.R.L. ""ECEPCION S/D: Tengo el agrado de dirigirme a Usted en los autos caratulados: IMARESCO GRACIELA SILVIA C/RIOBO SEBASTIAN LEANDRO S/COBRO DE HONORARIOSW (Exptc. N*17513/2025), que tramitan por ante cl juzgado de Familia N*5 de Avellaneda, del Dpto. Jud. Avellaneda/Lanú..."
159,529790,3,persona,juez,0.378836,True,"circunstancias especiales. No podrán establecer Tecaudos que no estuvieran autorizados por ley. Los oficios librados deberán ser"" recibidos obligatoriamente a su presentación. El juez deberá aplicar sanciones conminatorias progresivas en el supuesto de atraso injustificado en las contestaciones..."
160,542140,1,persona,GONZALES GONZALES JUANA,0.378142,True,"nder por intereses y costas, debiendo la entidad financiera depositar dichos montos en la cuenta N* 9958412379, Sucursal: 0089, Producto; 99, Denominación: GUTIERREZ MONTES ELOY C/GONZALES GONZALES JUANA Poder Judicial de la Nación juzgado CIVIL 92 JACQUELINE S/DIVORCIO, abierta en el banco de l..."
161,536741,5,persona,CAETANO E,0.374051,True,"R 2000 RECIBIDO Validez desconocidaFirma válida Digitally signedsby4/EBASTIAN JULIO MARTI fe) Date: 2025.12.16""12:03:29 ART + ue P se ás GS ñ - € 32 -obiiion: sb ae sb st AAA El e CAETANO E 1) emos se ner 5 enla erecta o tinta nes 05.8 20 0052 5 UD mode: ""añ Él ios en en om al e ramdágoni"


**celda para mejorar la plantilla de revisión**

Tu plantilla actual tiene las 440 filas, pero no incluye las columnas nuevas:

`formato_valido_label`
`requiere_revision_manual`

In [ ]:
# Crear plantilla de revisión mejorada con columnas de validación

df_revision_mejorada = df_entidades[
    ~df_entidades["entidad_label"].isin(["SIN_ENTIDADES", "SIN_TEXTO", "ERROR"])
].copy()

# Asegurarse de que existan las columnas nuevas
if "formato_valido_label" not in df_revision_mejorada.columns:
    print("Falta la columna formato_valido_label. Ejecutá primero la celda mejorada de validación.")

if "requiere_revision_manual" not in df_revision_mejorada.columns:
    print("Falta la columna requiere_revision_manual. Ejecutá primero la celda mejorada de validación.")

columnas_revision_mejorada = [
    "id",
    "modelo",
    "labels_set",
    "threshold",
    "estrategia",
    "chunk_id",
    "entidad_label",
    "entidad_texto",
    "score",
    "formato_valido_label",
    "requiere_revision_manual",
    "texto_contexto",
]

df_revision_mejorada = df_revision_mejorada[columnas_revision_mejorada].copy()

df_revision_mejorada["correcto_manual"] = ""
df_revision_mejorada["label_correcto_manual"] = ""
df_revision_mejorada["valor_correcto_manual"] = ""
df_revision_mejorada["observaciones"] = ""

# Ordenar primero lo que requiere revisión
df_revision_mejorada = df_revision_mejorada.sort_values(
    ["requiere_revision_manual", "entidad_label", "score"],
    ascending=[False, True, True]
).reset_index(drop=True)

REVISION_MEJORADA_PATH = OUTPUT_DIR / "revision_supervisada_gliner_embargos_15_mejorada.csv"

df_revision_mejorada.to_csv(
    REVISION_MEJORADA_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Plantilla mejorada guardada:", REVISION_MEJORADA_PATH)
print("Filas:", len(df_revision_mejorada))

display(df_revision_mejorada.head(40))

Plantilla mejorada guardada: /content/outputs_03_gliner_embargos/revision_supervisada_gliner_embargos_15_mejorada.csv
Filas: 440


,id,modelo,labels_set,threshold,estrategia,chunk_id,entidad_label,entidad_texto,score,formato_valido_label,requiere_revision_manual,texto_contexto,correcto_manual,label_correcto_manual,valor_correcto_manual,observaciones
0,531044,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,4,CBU,30-70721665-0,0.500732,False,True,"30-70721665-0) a la orden del juzgado y como perteneciente a estos autos, dentro del plazo de cinco días hábiles posteriores a la liquidación de los haberes correspondientes, por lo que, se ord",,,,
1,547358,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,5,CBU,141038119503,0.561555,False,True,"CVU: 0000003100087376033711 Número de operación de Mercado Pago 134597534609 Scanned with (BS CamScanner Detalle de operación Creada el 12 de enero - 10:16 hs Número de operación 141038119503 (bh () transferencia de dinero $ 150.000,00 por ""Varios"" [y] Aprobada (9) transferencia de Sebastian Le...",,,,
2,531044,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,3,CBU,01401505275049509868834,0.850708,False,True,"ROBERTO C/ ROLON GONZALEZ JULIO CESAR S/ COBRO DE HONORARIOS - Número: 6527 informando cuenta judicial N"" 027-509688/3 y CBU N* 01401505275049509868834. CUIT Poder Judicial 30- 70721665-0. El auto ordenatorio dice ""Avellaneda,17 de noviembre de 2025. Conforme lo pedido y hasta cubrir la suma de ...",,,,
3,546231,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,3,CBU,2204 36324485570295202512231 00909208 0290075900204069109960,0.851950,False,True,"e que cumpla con el embargo ordenado el día 30/9/25, hágase saber a la entidad oficiada que deberá depositar las sumas retenidas en la cuenta judicial: T"" 691 F* 996 DV 0, con CBU 2204 36324485570295202512231 00909208 0290075900204069109960. A ial in líbrese oficio digital. Hágase saber a la par...",,,,
4,531044,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,4,CBU,CBU,0.941261,False,True,"n de los haberes correspondientes, por lo que, se ordena proceder a la apertura de la cuenta judicial en cuestión, fecho la parte, deberá consignar los datos de la misma, número y CBU, en el oficio al empleador. A tales fines líbrese por secretaría en forma electrónica la pieza pertinente. Fecho...",,,,
5,531044,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,2,CBU,014015052750,0.943527,False,True,"EMA CORTE DE JUSTICIA - NOTIFICACIONES ELECTRÓNICAS ROBERTO C/ ROLON GONZALEZ JULIO CESAR S/ COBRO DE HONORARIOS - Número: 6527 informando cuenta judicial N"" 027-509688/3 y CBU N* 014015052750",,,,
6,529790,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,CBU,01100402-500990,0.948257,False,True,"IT. 20235539897, conforme informara en autos con fecha 04/11/25, a la cuenta de autos abierta en el banco Nación, Sucursal San Martin, que se identifica con el N? 9902829493, CBU. 01100402-500990",,,,
7,546250,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,4,CUIL,e.1.,0.366240,False,True,"expediente separado. Cuando se tratare de la inscripción de la transferencia de dominio en el Registro de la Propiedad, los oficios que se libren a Obras Sanitarias de la Nación (e.1.) al ente prestador de ese servicio y al Gobierno de la Ciudad de Buenos Aires o Municipio de que se trate, cont...",,,,
8,547358,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,4,CUIL,CUIT/CUIL,0.651721,False,True,oviembre de 2025 a las 14:54 hs $ 11.600 Motivo: Varios * De Sebastian Leandro Riobo CUIT/CUIL: 20-27755752-6 Mercado Pago CVU: 0000003100094121451599 o Para Vanesa Natalia Masimo CUIT/CUIL;,,,,
9,538081,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,6,CUIT,CAlMQO,0.371668,False,True,es.scba.gov.ar/verificarnotificacion.aspx y complete con el siguente codigo 8AD2HPOY | 1 MAR 200 RECIBIDO Datos del Firmante o Firmado por.: SPADA Juan Angel (CUIL 23312991829) '- CAlMQO.ccconono: NO DISPONIBLE Cert. Serial ++: 2832202425276362862427019879230250304058876116 Emisor.: Autoridad Ce...,,,,


**celda para separar dos archivos**

Uno para confiables y otro para revisar:

In [ ]:
df_confiables = df_revision_mejorada[
    df_revision_mejorada["requiere_revision_manual"] == False
].copy()

df_a_revisar = df_revision_mejorada[
    df_revision_mejorada["requiere_revision_manual"] == True
].copy()

CONFIABLES_PATH = OUTPUT_DIR / "entidades_multi_pii_v1_confiables.csv"
A_REVISAR_PATH = OUTPUT_DIR / "entidades_multi_pii_v1_a_revisar.csv"

df_confiables.to_csv(CONFIABLES_PATH, index=False, encoding="utf-8-sig")
df_a_revisar.to_csv(A_REVISAR_PATH, index=False, encoding="utf-8-sig")

print("Entidades confiables:", CONFIABLES_PATH, "| filas:", len(df_confiables))
print("Entidades a revisar:", A_REVISAR_PATH, "| filas:", len(df_a_revisar))

Entidades confiables: /content/outputs_03_gliner_embargos/entidades_multi_pii_v1_confiables.csv | filas: 277
Entidades a revisar: /content/outputs_03_gliner_embargos/entidades_multi_pii_v1_a_revisar.csv | filas: 163


**Guardar ambos CSV**

In [ ]:
CONFIABLES_PATH = OUTPUT_DIR / "entidades_multi_pii_v1_confiables.csv"
REVISION_PATH = OUTPUT_DIR / "entidades_multi_pii_v1_a_revisar.csv"

df_confiables.to_csv(CONFIABLES_PATH, index=False, encoding="utf-8-sig")
df_a_revisar.to_csv(REVISION_PATH, index=False, encoding="utf-8-sig")

print("Entidades confiables:", CONFIABLES_PATH)
print("Entidades a revisar:", REVISION_PATH)

Entidades confiables: /content/outputs_03_gliner_embargos/entidades_multi_pii_v1_confiables.csv
Entidades a revisar: /content/outputs_03_gliner_embargos/entidades_multi_pii_v1_a_revisar.csv


**Celda 14 — Verificación auxiliar con regex**

Esto no reemplaza GLiNER. Sirve para ver si había datos obvios en el texto y GLiNER los detectó o no.

In [ ]:
REGEX_PATTERNS = {
    "dni": r"\b(?:DNI|D\.N\.I\.|DOCUMENTO|DOC\.?)\D{0,20}\d{7,8}\b|\b\d{7,8}\b",
    "cuil_cuit": r"\b(?:CUIL|CUIT|C\.U\.I\.L\.|C\.U\.I\.T\.)\D{0,20}(?:20|23|24|27|30|33|34)[\-\s]?\d{8}[\-\s]?\d\b|\b(?:20|23|24|27|30|33|34)[\-\s]?\d{8}[\-\s]?\d\b",
    "cbu_cvu": r"\b(?:CBU|CVU)\D{0,20}\d(?:[\s\-]?\d){21}\b|\b\d(?:[\s\-]?\d){21}\b",
    "monto": r"(?:\$|pesos|importe|monto)\s*[\d\.\,]+",
    "expediente": r"\b(?:expediente|expte|exp\.|nro\.?|n°|numero)\D{0,30}\d+[\w\-\/]*",
}


def regex_tiene(texto, pattern):
    if not isinstance(texto, str):
        texto = str(texto)

    return bool(re.search(pattern, texto, flags=re.IGNORECASE))


df_regex = df[[ID_COL, TEXT_COL]].copy()
df_regex = df_regex.rename(columns={ID_COL: "id"})

for nombre, pattern in REGEX_PATTERNS.items():
    df_regex[f"regex_tiene_{nombre}"] = df_regex[TEXT_COL].apply(
        lambda x: regex_tiene(x, pattern)
    )

display(df_regex.head())

,id,texto_limpieza_avanzada,regex_tiene_dni,regex_tiene_cuil_cuit,regex_tiene_cbu_cvu,regex_tiene_monto,regex_tiene_expediente
0,536703,"(Ue\n\n413/26, 09-18 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA- NOTIFICACIONES ELECTRÓNICAS\nUsuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar\nOrganismo: TRIBUNAL DEL TRABAJO N? 5 - QUILMES\ncarátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LA...",True,True,False,True,True
1,542140,"Poder Judicial de la Nación\njuzgado CIVIL 92\n\n29921/2018\n\nIncidente N* 1 - ACTOR: GONZALES\nGONZALES, JUANA JAQUELINE Y OTRO\nDEMANDADO: GUTIERREZ MONTES, ELOY\nES mercadO E ECUCION DE ALIMENTOS - INCIDENTE\nlibre\n\n1.3 MAR 2026 nn\npPCION CABA, 09 de Febrero de 2026.-\nRECE\n\nSEÑOR DIREC...",True,True,False,True,True
2,546228,"a 1+\n\nPoder Judicial de la Nación\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\n\ns/ALIMENTOS\npoa Buenos Aires, de marzo de 2026.- MEG\n\nTy Z OFICIO REITERATORIO\n\nMercado Pago\nAssetmanagement SA\n\nS / D\n\nTengo el agrado de dirigirme a Ud. en los autos...",True,False,True,True,True
3,531044,"18/2/26, 10:49 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA - NOTIFICACIONES ELECTRÓNICAS dl 2)\n\nscba.gov.al\n\n7 Presentaciones y\n\nÑ Notificaciones Electrónicas\nA SUPREMA CORTE DE JUSTICIA\n\n<< aaa PROVINCIA DE BUENOS AIRES\n\nIsuario Conectado:Omar Roberto Garcia - Acceso...",True,True,False,True,True
4,546230,"+ 19 MAR 2026 |\n\n1\n\nRECEPCION\n\nPoder Judicial de la Nación\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\ns/ALIMENTOS\n\nBuenos Aires, de marzo de 2026.- MEG\nOFICIO REIFERATORIO\n\nMercado Pago\nAssetmanagement SA\nS / D\n\nTengo el agrado de dirigirme a ...",True,False,True,True,True


In [ ]:
def gliner_detecto_label(df_entidades, doc_id, patrones_label):
    subset = df_entidades[
        (df_entidades["id"] == doc_id)
        & (~df_entidades["entidad_label"].isin(["SIN_ENTIDADES", "SIN_TEXTO", "ERROR"]))
    ]

    labels = subset["entidad_label"].fillna("").str.lower().tolist()

    for label in labels:
        for patron in patrones_label:
            if patron in label:
                return True

    return False


LABEL_PATTERNS = {
    "dni": ["dni"],
    "cuil_cuit": ["cuil", "cuit"],
    "cbu_cvu": ["cbu", "cvu", "cuenta"],
    "monto": ["monto"],
    "expediente": ["expediente"],
}

for nombre, patrones in LABEL_PATTERNS.items():
    df_regex[f"gliner_detecto_{nombre}"] = df_regex["id"].apply(
        lambda doc_id: gliner_detecto_label(df_entidades, doc_id, patrones)
    )

display(df_regex)

,id,texto_limpieza_avanzada,regex_tiene_dni,regex_tiene_cuil_cuit,regex_tiene_cbu_cvu,regex_tiene_monto,regex_tiene_expediente,gliner_detecto_dni,gliner_detecto_cuil_cuit,gliner_detecto_cbu_cvu,gliner_detecto_monto,gliner_detecto_expediente
0,536703,"(Ue\n\n413/26, 09-18 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA- NOTIFICACIONES ELECTRÓNICAS\nUsuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar\nOrganismo: TRIBUNAL DEL TRABAJO N? 5 - QUILMES\ncarátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LA...",True,True,False,True,True,True,True,False,True,True
1,542140,"Poder Judicial de la Nación\njuzgado CIVIL 92\n\n29921/2018\n\nIncidente N* 1 - ACTOR: GONZALES\nGONZALES, JUANA JAQUELINE Y OTRO\nDEMANDADO: GUTIERREZ MONTES, ELOY\nES mercadO E ECUCION DE ALIMENTOS - INCIDENTE\nlibre\n\n1.3 MAR 2026 nn\npPCION CABA, 09 de Febrero de 2026.-\nRECE\n\nSEÑOR DIREC...",True,True,False,True,True,True,True,False,True,True
2,546228,"a 1+\n\nPoder Judicial de la Nación\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\n\ns/ALIMENTOS\npoa Buenos Aires, de marzo de 2026.- MEG\n\nTy Z OFICIO REITERATORIO\n\nMercado Pago\nAssetmanagement SA\n\nS / D\n\nTengo el agrado de dirigirme a Ud. en los autos...",True,False,True,True,True,True,False,True,True,True
3,531044,"18/2/26, 10:49 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA - NOTIFICACIONES ELECTRÓNICAS dl 2)\n\nscba.gov.al\n\n7 Presentaciones y\n\nÑ Notificaciones Electrónicas\nA SUPREMA CORTE DE JUSTICIA\n\n<< aaa PROVINCIA DE BUENOS AIRES\n\nIsuario Conectado:Omar Roberto Garcia - Acceso...",True,True,False,True,True,True,True,True,True,True
4,546230,"+ 19 MAR 2026 |\n\n1\n\nRECEPCION\n\nPoder Judicial de la Nación\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\ns/ALIMENTOS\n\nBuenos Aires, de marzo de 2026.- MEG\nOFICIO REIFERATORIO\n\nMercado Pago\nAssetmanagement SA\nS / D\n\nTengo el agrado de dirigirme a ...",True,False,True,True,True,True,False,True,True,True
5,546231,"10)\n\n19 MAR 2026\n\nOFICIO\nBuenos Aires 23 de diciembre de 2025.\n\nAl Señor Gerente de Mercado Pago Asset Management S.A.\nCurr 33-71862856-9. e\n\n4\n\nTengo el agrado de dirigirme a Usted en los autos\ncaratulados: ""BBVABanco Francés S.A. C/ CARUSO Gustavo\nDaniel S/ Ejecutivo"" (expediente...",True,True,True,True,True,True,True,True,True,True
6,536741,"OFICIO\n\nembargo DE CUENTAS\n\n1.0 MAR 206\n\nBuenos Aires, de diciembre de 2025\nRECIBIDO\nAl Señor Gerente de Mercado Pago Asset Management S.A.\nCUIT 33-71862856-9. (Mercado Fondos)\n\nSs / D\n\nTengo el agrado de\ndirigirme a Usted en los autos caratulados: ""BBVABanco Francés S.A. C/\nLOBO ...",True,True,True,True,True,True,True,True,True,True
7,547358,"19/3/26, 17:47. about:blank\nDATOS NOTIFICACION ELECTRONICA\nUsuario conectado: MARESCO Graciela Silvia\n\nOrganismo: juzgado DE FAMILIA N? 5 - AVELLANEDA\ncarátula: MARESCO GRACIELA SILVIA C/RIOBO SEBASTIAN LEANDRO S/COBRO DE: HONORARIOS\n\nNúmero de causa: AL-17513-2025\n\nTipo de notificación...",True,True,True,True,True,False,True,True,True,True
8,529790,"0.3 MAR 2006 OFICIO\n\nRECIBIDO\n\nSan Martin, Y. de: YIALTO de2026.-\n\ndar do\n\nAl Sr. Gerente del\nMercado Libre S.R.L. E\n\nPresente.-\n\nTengo el agrado de dirigirme a Ud. en los autos caratulados:\n""SINDICATO UNICO DE TRABAJADORES DE PELUQUERIAS, ESTETICA\nY AFINES DE BUENOS AIRES c/ HERR...",True,True,True,True,True,False,True,True,True,True
9,546250,"cado\n\nUNS\npus is,\n\n2 MAR 2026\nRECEPCION\n\n1\n\nPoder Judicial de la Nación\n\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\ns/ALIMENTOS\n\nBuenos Aires, de marzo de 2026.- MEG\n\nOFICIO REITERATORIO\nMercado Libre SRL\nS ¡/ D\n\nTengo el agrado de dirigir...",True,False,True,True,True,True,True,True,True,True


**Celda 15 — Crear plantilla de revisión supervisada**

In [ ]:
df_revision = df_entidades[
    ~df_entidades["entidad_label"].isin(["SIN_ENTIDADES", "SIN_TEXTO", "ERROR"])
].copy()

columnas_revision = [
    "id",
    "modelo",
    "labels_set",
    "threshold",
    "estrategia",
    "chunk_id",
    "entidad_label",
    "entidad_texto",
    "score",
    "texto_contexto",
]

df_revision = df_revision[columnas_revision].copy()

df_revision["correcto_manual"] = ""
df_revision["label_correcto_manual"] = ""
df_revision["valor_correcto_manual"] = ""
df_revision["observaciones"] = ""

REVISION_PATH = OUTPUT_DIR / "revision_supervisada_gliner_embargos_15.csv"

df_revision.to_csv(
    REVISION_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Plantilla de revisión guardada:", REVISION_PATH)
print("Filas para revisar:", len(df_revision))

display(df_revision.head(30))

Plantilla de revisión guardada: /content/outputs_03_gliner_embargos/revision_supervisada_gliner_embargos_15.csv
Filas para revisar: 440


,id,modelo,labels_set,threshold,estrategia,chunk_id,entidad_label,entidad_texto,score,texto_contexto,correcto_manual,label_correcto_manual,valor_correcto_manual,observaciones
0,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,persona,DOS SANTOS CARLOS ROBERTO,0.966278,"(Ue 413/26, 09-18 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA- NOTIFICACIONES ELECTRÓNICAS Usuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar Organismo: TRIBUNAL DEL TRABAJO N? 5 - QUILMES carátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO E...",,,,
1,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,caratula,DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO,0.921589,USTICIA- NOTIFICACIONES ELECTRÓNICAS Usuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar Organismo: TRIBUNAL DEL TRABAJO N? 5 - QUILMES carátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO Número de causa: 18645 Tipo de notificación: OFICIO R...,,,,
2,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,numero de expediente,18645,0.716898,O - 23137661349Q notificaciones.scba.govar Organismo: TRIBUNAL DEL TRABAJO N? 5 - QUILMES carátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO Número de causa: 18645 Tipo de notificación: OFICIO REQUIRIENDO Destinatarios: 23137661349 notificaciones.scba.govar Fecha notificaci...,,,,
3,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,destino de transferencia,23137661349 notificaciones.scba.govar,0.507768,DEL TRABAJO N? 5 - QUILMES carátula: DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO Número de causa: 18645 Tipo de notificación: OFICIO REQUIRIENDO Destinatarios: 23137661349 notificaciones.scba.govar Fecha notificación: 27/2/2026 08:27:57 Alta o disponibilidad 27/2/202608:28:06...,,,,
4,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,persona,ZACARIAS Andrea Marcela,0.972130,137661349 notificaciones.scba.govar Fecha notificación: 27/2/2026 08:27:57 Alta o disponibilidad 27/2/202608:28:06 10 AR 2008 Firma digital: Firma válida Firmado y Notificado por: ZACARIAS Andrea Marcela. JUEZ - Certifigado 08:28:05 Elrmado por: ZACARIAS Andrea Marcela. JUEZ - Certificado Correc...,,,,
6,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,persona,DOS SANTOS Carlos Roberto,0.990230,rmado y Notificado por: ZACARIAS Andrea Marcela. JUEZ - Certifigado 08:28:05 Elrmado por: ZACARIAS Andrea Marcela. JUEZ - Certificado Correcto. Fecha de Firma: 27/02/2026 08:28:04 DOS SANTOS Carlos Roberto. - Certificado Correcto. Fecha de Firma: 25/02/2026 13:55:01 OFICIO DE embargo AMERCADO PA...,,,,
7,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,0,caratula,DEFINA STELLA MARIS,0.558497,"S Carlos Roberto. - Certificado Correcto. Fecha de Firma: 25/02/2026 13:55:01 OFICIO DE embargo AMERCADO PAGO SID.- Tengo el agrado de dirigirme a Usted en los autos caratulados "":DEFINA STELLA MARIS",,,,
9,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,1,numero de expediente,N* Causa: 18645,0.883877,"26 13:55:01 OFICIO DE embargo AMERCADO PAGO SID.- Tengo el agrado de dirigirme a Usted en los autos caratulados "":DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO"" - N* Causa: 18645, que tramita ante el Tribunal de Trabajo N* 5 del Departamento Judicial de Quilmes, a cargo del Dr. ...",,,,
10,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,1,persona,Dr. Mario Daniel Stolarczyk,0.977235,""":DEFINA STELLA MARIS C/ ARAUJO ARAUJO LAZARO EDUARDO S/ DESPIDO"" - N* Causa: 18645, que tramita ante el Tribunal de Trabajo N* 5 del Departamento Judicial de Quilmes, a cargo del Dr. Mario Daniel Stolarczyk, secretaría a cargo de la Dra. María Fernanda Sartori, sito en calle Alvear nro. 838 per...",,,,
11,536703,urchade/gliner_multi_pii-v1,embargo_simple,0.35,chunks,1,persona,Dra. María Fernanda Sartori,0.971395,"S/ DESPIDO"" - N* Causa:

**Celda 16 — Descargar resultados**

In [ ]:
files.download(str(ENTIDADES_PATH))
files.download(str(COMPARATIVA_PATH))
files.download(str(REVISION_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("/content/outputs_03_gliner_embargos/entidades_multi_pii_v1_confiables.csv")
files.download("/content/outputs_03_gliner_embargos/entidades_multi_pii_v1_a_revisar.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusión de la prueba de modelos GLiNER sobre embargos

En esta etapa se trabajó únicamente con la muestra de 15 embargos previamente seleccionados como los más legibles luego de la limpieza avanzada de OCR. El objetivo fue evaluar la extracción de entidades relevantes para embargos, principalmente: monto, destino de transferencia, número de expediente, carátula, personas, DNI, CUIL, CUIT, CBU y banco.

Para evitar el problema detectado en pruebas anteriores, donde los textos largos podían ser truncados internamente por GLiNER, se utilizó una estrategia de procesamiento por chunks. Cada documento fue dividido en fragmentos de hasta 300 tokens, con un overlap de 50 tokens, para reducir la pérdida de información entre cortes.

Se probaron dos modelos principales:

1. `gliner-community/gliner_medium-v2.5`
2. `urchade/gliner_multi_pii-v1`

El primer modelo, `gliner-community/gliner_medium-v2.5`, logró detectar entidades en los 15 embargos, pero generó una cantidad muy alta de resultados: 976 entidades. En la revisión inicial se observaron varios falsos positivos, especialmente en etiquetas como DNI, CUIL y CUIT. Por ejemplo, algunos lugares o instituciones fueron clasificados incorrectamente como identificadores personales. Esto indica que, aunque el modelo puede detectar muchas entidades, también introduce bastante ruido para esta tarea específica.

El segundo modelo, `urchade/gliner_multi_pii-v1`, mostró un comportamiento más conveniente para el caso de uso. Este modelo detectó 440 entidades en total, cubriendo también los 15 embargos y sin errores de ejecución. El score promedio fue de aproximadamente 0.776. A diferencia del modelo anterior, la salida fue más controlada y más enfocada en entidades útiles para embargos, especialmente personas, montos, CUIT, DNI, bancos, números de expediente y carátulas.

Además, se agregaron validaciones posteriores para marcar entidades sospechosas o que requieren revisión manual. Estas reglas permitieron separar entidades con formato aparentemente válido de aquellas que necesitan revisión, especialmente en casos como CBU mal detectados, destinos de transferencia dudosos, bancos genéricos, carátulas parciales o personas con cargos pegados.

Como resultado, el modelo `urchade/gliner_multi_pii-v1` fue el más conveniente en esta prueba inicial, ya que redujo considerablemente el ruido en comparación con `gliner-community/gliner_medium-v2.5` y produjo entidades más alineadas con las necesidades del proyecto.

La configuración preliminar más útil fue:

* Modelo: `urchade/gliner_multi_pii-v1`
* Threshold: `0.35`
* Labels: `embargo_simple`
* Estrategia: `chunks`
* Máximo por chunk: `300 tokens`
* Overlap: `50 tokens`

De todos modos, los resultados no deben considerarse definitivos sin revisión humana. La extracción automática funciona como una primera capa de detección, pero todavía es necesario validar manualmente las entidades extraídas, especialmente las relacionadas con CBU, destino de transferencia, carátulas parciales y datos personales detectados en contextos ambiguos.

Archivos generados en esta etapa:

* `entidades_gliner_embargos_15.csv`
* `comparativa_modelos_gliner_embargos_15.csv`
* `revision_supervisada_gliner_embargos_15.csv` es una tabla para que una persona revise manualmente las entidades detectadas por GLiNER.
* `revision_supervisada_gliner_embargos_15_mejorada.csv`
* `entidades_multi_pii_v1_confiables.csv`
* `entidades_multi_pii_v1_a_revisar.csv`

Como próximo paso, se recomienda revisar manualmente las entidades marcadas como dudosas, validar una muestra de las entidades confiables y luego decidir si se mantiene el threshold en `0.35` o si conviene probar una segunda corrida con threshold más alto, por ejemplo `0.45`, para reducir todavía más los falsos positivos.
